In [1]:
import torch
import sys
import gc
print(sys.version)
print(f"PyTorch Version: {torch.__version__}")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(torch.cuda.get_device_name(0))
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print(torch.cuda.is_bf16_supported())

3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
PyTorch Version: 2.12.0+cu126
True
1
CUDA Version: 12.6
NVIDIA GeForce RTX 4080 Laptop GPU
True


# LLM Reliability Case Study - Qwen2.5-0.5B-Instruct

#### **Challenge**
Large Language Models generate probabilistic outputs that cannot always be trusted in operational environments, leading to token drift, logical inconsistencies, and unverified execution paths.

#### **Approach**
The experiment implemented an inference-time verification pipeline incorporating:

* **Multi-Sample Response Generation:** Branching generations into $N$ parallel trajectories to explore the model's latent solution space.
* **Cross-Candidate Semantic Evaluation:** Programmatically checking alignment across different reasoning outputs.
* **Consensus Estimation Across Outputs:** Treating confidence as an emergent property of sampling density rather than a scalar token probability.
* **Contradiction Detection:** Analyzing reasoning paths side-by-side to catch logic decay or divergent facts early.
* **Dynamic Uncertainty Scoring:** Flagging unstable distributions across candidate outputs to dynamically calculate system risk.
* **Ground-Truth Validation:** Injecting rigid constraints to check generations against known, deterministic facts.
* **Human-in-the-Loop Escalation:** Safely routing execution to a manual gatekeeper if the consensus metric drops below the stability threshold.

In [2]:
import torch
import numpy as np
import re
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder

# Setup Models
baseline_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(baseline_model_name, trust_remote_code=True)
baseline_model = AutoModelForCausalLM.from_pretrained(baseline_model_name, device_map="auto")
device = baseline_model.device

# Load Cross-Encoder on CPU to preserve precious GPU VRAM
relevance_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
from sentence_transformers import SentenceTransformer

# Load the compact bi-encoder on CPU to handle the dense semantic consensus axes
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

prompts = [
    "What is AI?",
    "Tell me something interesting about Albert Einstein.",
    "Tell me something about Large Language Models.",
    "What is geometry? Explain it step by step.",
    "Explain the concept of entropy in simple terms.",
    "Tell me something about Jean Baudrillard.",
    "Who was David Hilbert?",
    "Give me three facts about London.",
    "Tell me something about love.",
    "Why did Albert Einstein invent the internet in 1969?"
]

# HARD FILTERS
def fatal_filter(samples):
    """
    Only removes genuinely unusable generations.
    Everything else is preserved for ranking.
    """
    passed_samples = []

    for s in samples:
        text = s["text"].strip()

        # FATAL: empty or trivial output
        if len(text) < 10:
            continue

        # FATAL: invalid or corrupted perplexity
        if s["ppl"] is None or s["ppl"] > 1e6:
            continue

        # FATAL: tokenization / byte explosion
        char_len = len(text)
        token_len = len(s.get("tokens", []))
        if token_len == 0 or (char_len / max(token_len, 1)) < 1.5:
            continue

        # FATAL: extreme repetition collapse
        tokens = text.split()
        if len(tokens) >= 20:
            freq = Counter(tokens)
            top_ratio = freq.most_common(1)[0][1] / len(tokens)
            if top_ratio > 0.70:
                continue

        passed_samples.append(s)

    return passed_samples


import random
import numpy as np
import torch

def generate_n(model, tokenizer, inputs, prompt_len, N=5, temp=0.4, top_p=0.9):
    import random
    import numpy as np
    import torch

    samples = []

    # generate independent seeds
    seeds = [random.randint(0, 2**32 - 1) for _ in range(N)]

    for seed in seeds:
        # isolate randomness
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=500,
                do_sample=True,
                temperature=temp,
                top_p=top_p,
                num_return_sequences=1,
            )

        # decode only assistant continuation
        gen_tokens = output[0][prompt_len:]
        text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

        # compute conditional NLL on generated sequence only
        with torch.no_grad():
            outputs = model(output)
            logits = outputs.logits

            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = output[:, 1:].contiguous()

            # only evaluate generated region
            gen_shift_start = prompt_len - 1
            gen_logits = shift_logits[0, gen_shift_start:]
            gen_labels = shift_labels[0, gen_shift_start:]

            loss = torch.nn.functional.cross_entropy(
                gen_logits.view(-1, gen_logits.size(-1)),
                gen_labels.view(-1),
                reduction="mean"
            )

            nll = float(loss.item())
            ppl = float(torch.exp(loss).item())

        samples.append({
            "text": text,
            "ppl": ppl,
            "nll": nll,
            "seed_used": seed,
            "tokens": gen_tokens
        })

    return samples

C:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████| 103/103 [00:00<00:00, 27487.01it/s]


1. We use a test suite of 10 prompts. The final prompt ("Why did Albert Einstein invent the internet in 1969?") is intentionally nasty. It injects a completely false premise designed to provoke the LLM into a hallucination, letting us test how well the downstream pipeline catches a model under pressure
2. generate_n - While it is tempting to generate the $N$ responses in a single batch using standard engine arguments, batching often shares a random seed state that reduces output variety. We took special care to loop generations sequentially and isolate seeds for each pass, forcing the model to explore a wider variety of generated outputs
3. This filter does not judge the quality or meaning of the variants. Its sole purpose is to quickly reject clear rubbish - like empty text, extreme repetition loops, or broken decoding states while preserving all other responses to look at later

## Optimization Plane
Linguistic & NLI Pipeline Foundations
Before running the evaluation loop, this cell initializes the dual-engine scoring architecture (spaCy + DeBERTa-v3) required to compute the trajectory weights.

1. Why spaCy (en_core_web_sm)?
We use spaCy for deterministic grammatical verification. Instead of relying on an LLM to guess if an output is structured correctly, spaCy performs shallow NLP parsing (tokenization, sentence segmentation, and part-of-speech tagging). This allows us to algorithmically flag structural collapse, such as truncated sentences or trailing clauses, before expensive semantic scoring begins.

2. Why DeBERTa-v3-base-mnli-fever-anli for NLI?
Natural Language Inference (NLI) is our engine for factuality and logic validation. This specific DeBERTa-v3 model is fine-tuned on three major benchmarks (MNLI, FEVER, and ANLI), making it exceptionally robust at detecting zero-shot semantic relationships between a prompt's implicit assumptions and the generated outputs. It classifies candidate trajectories into three explicit states: Entailment (logical follow-through), Neutral, or Contradiction (hallucination/drift).

3. Analyzing branch_disagreement
The pipeline evaluates divergence across the parallel streams by measuring how much candidate outputs conflict with each other semantically. High semantic disagreement indicates that the 0.5B model's latent space is highly unstable for that specific prompt, serving as a signature that the model is guessing rather than converging.

4. The Need for dynamic_weights
Not all prompts are created equal. A factual query requires strict NLI constraints, while a creative query requires grammatical fluency. dynamic_weights adjust the scoring calculus on the fly based on the prompt's profile—balancing the trade-offs between strict factual alignment, vocabulary diversity, and structural formatting requirements.

5. Computing contradiction_penalties
Contradiction penalties are applied as a heavy mathematical down-regulation (negative energy multiplier) on any candidate trajectory where the NLI engine registers a high probability of a CONTRADICTION flag against known ground truths or premises. If a trajectory introduces a factual conflict, its score is aggressively tanked early, preventing it from winning the selection process.

In [3]:
# ===========================================================================
# ENERGY-BASED MULTI-OBJECTIVE TRAJECTORY RANKING FUNCTIONAL (STAGE 2)
# ===========================================================================

import re
import numpy as np
import spacy
from collections import Counter
from transformers import pipeline

print("Initializing Epistemic Stability Ranking Engine...")

nlp = spacy.load("en_core_web_sm")

nli_pipeline = pipeline(
    "text-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=0
)

# ===========================================================================
# BASIC SIGNALS
# ===========================================================================

def ppl_score(sample):

    ppl = sample.get("ppl")

    if ppl is None or ppl <= 0:
        return 0.0

    return 1.0 / (1.0 + np.log1p(ppl))


def completion_penalty(text):

    text = text.strip()

    if not text:
        return 0.0

    ends_cleanly = text[-1] in ".!?\"')]}”’"

    punct = 1.0 if ends_cleanly else 0.5

    wc = len(text.split())

    length = min(1.0, wc / 20.0)

    return punct * length


# ===========================================================================
# STRUCTURAL SENTINELS
# ===========================================================================

def structural_integrity(text):

    doc = nlp(text)

    patterns = []

    for sent in doc.sents:

        pos_pattern = tuple(
            token.pos_
            for token in sent
            if not token.is_punct
        )

        if len(pos_pattern) > 3:
            patterns.append(pos_pattern)

    if len(patterns) <= 1:
        return 1.0

    counts = Counter(patterns)

    repetition_ratio = (
        counts.most_common(1)[0][1] / len(patterns)
    )

    score = 1.0 - repetition_ratio

    return max(0.0, score)


# ===========================================================================
# NAMED ENTITY DENSITY
# ===========================================================================

def entity_density(text):

    doc = nlp(text)

    entity_labels = {
        "PERSON",
        "ORG",
        "GPE",
        "LOC",
        "DATE",
        "TIME",
        "CARDINAL",
        "ORDINAL",
        "QUANTITY",
        "MONEY",
        "PERCENT"
    }

    entity_tokens = 0

    for ent in doc.ents:

        if ent.label_ in entity_labels:
            entity_tokens += len(ent)

    total_tokens = max(len(doc), 1)

    density = entity_tokens / total_tokens

    return min(density * 5.0, 1.0)


# ===========================================================================
# CROSS-ENCODER SEMANTIC DRIFT
# ===========================================================================

def cohort_cross_encoder_consensus(texts, cross_encoder):

    n = len(texts)

    if n <= 1:
        return np.array([1.0]), 0.0

    similarity_matrix = np.zeros((n, n))

    for i in range(n):

        for j in range(i + 1, n):

            score = float(
                cross_encoder.predict(
                    [(texts[i], texts[j])]
                )[0]
            )

            similarity_matrix[i, j] = score
            similarity_matrix[j, i] = score

    min_s = similarity_matrix.min()
    max_s = similarity_matrix.max()

    similarity_matrix = (
        similarity_matrix - min_s
    ) / (max_s - min_s + 1e-8)

    np.fill_diagonal(similarity_matrix, 0)

    consensus = similarity_matrix.sum(axis=1)

    consensus /= max(n - 1, 1)

    semantic_drift = 1.0 - np.mean(consensus)

    return consensus, semantic_drift


# ===========================================================================
# CONTRADICTION AXIS
# ===========================================================================

def contradiction_score(samples):

    docs = [nlp(s["text"]) for s in samples]

    claims = []

    for d in docs:

        sent_claims = [
            s.text.strip()
            for s in d.sents
            if len(s.text.split()) > 5
        ][:3]

        claims.append(sent_claims)

    scores = []

    for i in range(len(claims)):

        contradictions = 0.0
        comparisons = 0

        for j in range(len(claims)):

            if i == j:
                continue

            for c1 in claims[i]:

                for c2 in claims[j]:

                    try:

                        result = nli_pipeline(
                            c1,
                            text_pair=c2
                        )[0]

                        if (
                            result["label"].lower()
                            == "contradiction"
                        ):
                            contradictions += result["score"]

                        comparisons += 1

                    except:
                        continue

        scores.append(
            contradictions / max(comparisons, 1)
        )

    return scores


# ===========================================================================
# NUMERIC CONSISTENCY
# ===========================================================================

def numeric_penalty(texts):

    numbers = [
        set(re.findall(r"\b\d+\b", t))
        for t in texts
    ]

    penalties = []

    for i in range(len(numbers)):

        disagreement = 0
        comparisons = 0

        for j in range(len(numbers)):

            if i == j:
                continue

            disagreement += len(
                numbers[i].symmetric_difference(
                    numbers[j]
                )
            )

            comparisons += 1

        penalties.append(
            disagreement / max(comparisons, 1)
        )

    maximum = max(penalties) if penalties else 1.0

    return [
        p / (maximum + 1e-8)
        for p in penalties
    ]


# ===========================================================================
# DYNAMIC WEIGHT ENGINE
# ===========================================================================

def calculate_dynamic_weights(semantic_drift):

    drift = np.clip(semantic_drift, 0.0, 1.0)

    weights = {

        "relevance": 0.30,

        "fluency": 0.20,

        "consensus": max(
            0.05,
            0.25 * (1.0 - drift)
        ),

        "structure": 0.10,

        "entity": 0.10,

        "contradiction": (
            0.10 + 0.30 * drift
        ),

        "numeric": 0.05
    }

    return weights


# ===========================================================================
# MAIN RANKING ENGINE
# ===========================================================================

def rank_samples_v3(
    prompt,
    samples,
    relevance_model
):

    if not samples:
        return []

    texts = [s["text"] for s in samples]

    fluency = [
        ppl_score(s)
        for s in samples
    ]

    completion = [
        completion_penalty(t)
        for t in texts
    ]

    structure = [
        structural_integrity(t)
        for t in texts
    ]

    entities = [
        entity_density(t)
        for t in texts
    ]

    consensus, semantic_drift = (
        cohort_cross_encoder_consensus(
            texts,
            relevance_model
        )
    )

    contradictions = contradiction_score(
        samples
    )

    numeric = numeric_penalty(texts)

    relevance = []

    for t in texts:

        score = float(
            relevance_model.predict(
                [(prompt, t)]
            )[0]
        )

        relevance.append(score)

    r_min = min(relevance)
    r_max = max(relevance)

    relevance = [

        (r - r_min)
        / (r_max - r_min + 1e-8)

        for r in relevance
    ]

    weights = calculate_dynamic_weights(
        semantic_drift
    )

    uncertainty_multiplier = np.exp(
        -2.0 * semantic_drift
    )

    safety_brake = (
        0.25
        if semantic_drift > 0.60
        else 1.0
    )

    ranked = []

    for i in range(len(samples)):

        score = (

            weights["relevance"]
            * relevance[i]

            + weights["fluency"]
            * fluency[i]

            + weights["consensus"]
            * consensus[i]

            + weights["structure"]
            * structure[i]

            + weights["entity"]
            * entities[i]

            - weights["contradiction"]
            * contradictions[i]

            - weights["numeric"]
            * numeric[i]
        )

        score *= completion[i]

        score *= uncertainty_multiplier

        score *= safety_brake

        if (
            contradictions[i] > 0.70
            or structure[i] < 0.20
        ):
            score = 0.0

        ranked.append({

            "score": float(score),

            "text": texts[i],

            "signals": {

                "relevance":
                    relevance[i],

                "fluency":
                    fluency[i],

                "consensus":
                    consensus[i],

                "semantic_drift":
                    semantic_drift,

                "structure":
                    structure[i],

                "entity_density":
                    entities[i],

                "contradiction":
                    contradictions[i],

                "numeric":
                    numeric[i],

                "uncertainty_multiplier":
                    uncertainty_multiplier,

                "safety_brake":
                    safety_brake
            }
        })

    ranked.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return ranked

Initializing Epistemic Stability Ranking Engine...


Loading weights: 100%|██████████████████████████████| 202/202 [00:00<00:00, 14365.03it/s]


## Stage 2: Matrix Loop

This cell runs the primary evaluation loop across your prompts, scoring and filtering the generated response variations along four distinct axes:

* **Fluency Axis:** Measures token-level language confidence using conditional perplexity metrics directly from the model.
* **Relevance Axis:** Evaluates topical alignment with the user's intent using cross-encoders.
* **Consensus Axis (`calculate_cohort_spread`):** Analyzes the cluster distance between text embeddings to map semantic drift and cluster spread across candidates. High divergence triggers adaptive risk multipliers or a human-in-the-loop (HITL) review loop.
* **Contradiction Axis:** Cross-checks active text variations side-by-side to penalize diverging claims or logic decay early.

The engine uses these axes to assign **dynamic weights** based on the prompt type, sorting the streams to surface a clear **Champion Branch** and a fallback **Contender Branch**.

In [4]:
# ===========================================================================
# HELPER FUNCTIONS
# ===========================================================================

import numpy as np

def classify_volatility(semantic_drift):

    if semantic_drift < 0.15:
        return "STABLE_CONVERGENCE"

    elif semantic_drift < 0.30:
        return "MILD_DIVERGENCE"

    elif semantic_drift < 0.50:
        return "MODERATE_FLUIDITY"

    elif semantic_drift < 0.70:
        return "HIGH_EPISTEMIC_CHAOS"

    else:
        return "SEVERE_GENERATIVE_INSTABILITY"


# ===========================================================================
# STAGE 2 MATRIX EXECUTION 
# ===========================================================================

pipeline_results = {}

for raw_prompt in prompts:

    print("\n" + "=" * 80)
    print(f"RUNNING INFERENCE SEARCH FOR:")
    print(f"'{raw_prompt}'")
    print("=" * 80)

    messages = [
        {
            "role": "user",
            "content": raw_prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(device)

    prompt_len = inputs.input_ids.shape[1]

    # ==========================================================
    # GENERATION
    # ==========================================================

    raw_candidates = generate_n(
        model=baseline_model,
        tokenizer=tokenizer,
        inputs=inputs,
        prompt_len=prompt_len,
        N=5,
        temp=0.4,
        top_p=0.9
    )

    filtered_candidates = fatal_filter(raw_candidates)

    print(
        f"-> Fatal Filters : "
        f"{len(filtered_candidates)}/{len(raw_candidates)} survived"
    )

    if not filtered_candidates:

        print("❌ No valid generations available.")
        continue

    # ==========================================================
    # STAGE 2 V3 RANKING
    # ==========================================================

    ranked_outputs = rank_samples_v3(
        prompt=raw_prompt,
        samples=filtered_candidates,
        relevance_model=relevance_model
    )

    if not ranked_outputs:

        print("❌ Ranking failure.")
        continue

    # ==========================================================
    # COHORT TELEMETRY
    # ==========================================================

    semantic_drift = ranked_outputs[0]["signals"]["semantic_drift"]

    cohort_spread = np.std(
        [
            c["signals"]["consensus"]
            for c in ranked_outputs
        ]
    )

    volatility = classify_volatility(
        semantic_drift
    )

    uncertainty_multiplier = (
        ranked_outputs[0]["signals"]
        ["uncertainty_multiplier"]
    )

    safety_brake = (
        ranked_outputs[0]["signals"]
        ["safety_brake"]
    )

    requires_human_review = (
        semantic_drift > 0.50
    )

    # ==========================================================
    # STORE PIPELINE RESULTS
    # ==========================================================

    pipeline_results[raw_prompt] = {

        "semantic_drift": semantic_drift,

        "cohort_spread": cohort_spread,

        "volatility": volatility,

        "uncertainty_multiplier":
            uncertainty_multiplier,

        "safety_brake":
            safety_brake,

        "requires_human_review":
            requires_human_review,

        "candidates": ranked_outputs[:3]
    }

    # ==========================================================
    # GLOBAL TELEMETRY REPORT
    # ==========================================================

    print("\n" + "-" * 80)
    print("STAGE 2 V3 EPISTEMIC TELEMETRY REPORT")
    print("-" * 80)

    print(
        f"Semantic Drift        : "
        f"{semantic_drift:.4f}"
    )

    print(
        f"Cohort Spread         : "
        f"{cohort_spread:.4f}"
    )

    print(
        f"Volatility State      : "
        f"{volatility}"
    )

    print(
        f"Uncertainty Multiplier: "
        f"{uncertainty_multiplier:.4f}x"
    )

    print(
        f"Safety Compression    : "
        f"{safety_brake:.2f}x"
    )

    print(
        f"HITL Escalation       : "
        f"{'YES' if requires_human_review else 'NO'}"
    )

    print("-" * 80)

    # ==========================================================
    # CHAMPION BRANCH
    # ==========================================================

    champion = ranked_outputs[0]

    s = champion["signals"]

    print(
        f"\n🥇 CHAMPION BRANCH "
        f"(Score: {champion['score']:.4f})"
    )

    print(
        f"Fluency               : "
        f"{s['fluency']:.4f}"
    )

    print(
        f"Relevance             : "
        f"{s['relevance']:.4f}"
    )

    print(
        f"Consensus             : "
        f"{s['consensus']:.4f}"
    )

    print(
        f"Structural Integrity  : "
        f"{s['structure']:.4f}"
    )

    print(
        f"Entity Density        : "
        f"{s['entity_density']:.4f}"
    )

    print(
        f"Contradiction         : "
        f"{s['contradiction']:.4f}"
    )

    print(
        f"Numeric Penalty       : "
        f"{s['numeric']:.4f}"
    )

    print(
        f"Semantic Drift        : "
        f"{s['semantic_drift']:.4f}"
    )

    print(
        f"Uncertainty Multiplier: "
        f"{s['uncertainty_multiplier']:.4f}x"
    )

    print(
        f"Safety Compression    : "
        f"{s['safety_brake']:.2f}x"
    )

    print("\n--- CHAMPION TEXT ---")
    print(champion["text"].strip())
    print("----------------------")

    # ==========================================================
    # CONTENDER BRANCH
    # ==========================================================

    if len(ranked_outputs) > 1:

        contender = ranked_outputs[1]

        s = contender["signals"]

        delta = (
            champion["score"]
            - contender["score"]
        )

        print(
            f"\n🥈 CONTENDER BRANCH "
            f"(Score: {contender['score']:.4f}, "
            f"Δ={delta:.4f})"
        )

        print(
            f"Fluency               : "
            f"{s['fluency']:.4f}"
        )

        print(
            f"Relevance             : "
            f"{s['relevance']:.4f}"
        )

        print(
            f"Consensus             : "
            f"{s['consensus']:.4f}"
        )

        print(
            f"Structural Integrity  : "
            f"{s['structure']:.4f}"
        )

        print(
            f"Entity Density        : "
            f"{s['entity_density']:.4f}"
        )

        print(
            f"Contradiction         : "
            f"{s['contradiction']:.4f}"
        )

        print(
            f"Numeric Penalty       : "
            f"{s['numeric']:.4f}"
        )

        print(
            f"Semantic Drift        : "
            f"{s['semantic_drift']:.4f}"
        )

        print(
            f"Uncertainty Multiplier: "
            f"{s['uncertainty_multiplier']:.4f}x"
        )

        print(
            f"Safety Compression    : "
            f"{s['safety_brake']:.2f}x"
        )

        print("\n--- CONTENDER TEXT ---")
        print(contender["text"].strip())
        print("-----------------------")

    # ==========================================================
    # THIRD BRANCH (OPTIONAL)
    # ==========================================================

    if len(ranked_outputs) > 2:

        third = ranked_outputs[2]

        print(
            f"\n🥉 THIRD BRANCH "
            f"(Score: {third['score']:.4f})"
        )

        print("\n--- THIRD BRANCH TEXT ---")
        print(third["text"].strip())
        print("--------------------------")

    print("\n" + "=" * 80)


RUNNING INFERENCE SEARCH FOR:
'What is AI?'
-> Fatal Filters : 5/5 survived


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



--------------------------------------------------------------------------------
STAGE 2 V3 EPISTEMIC TELEMETRY REPORT
--------------------------------------------------------------------------------
Semantic Drift        : 0.1862
Cohort Spread         : 0.0549
Volatility State      : MILD_DIVERGENCE
Uncertainty Multiplier: 0.6891x
Safety Compression    : 1.00x
HITL Escalation       : NO
--------------------------------------------------------------------------------

🥇 CHAMPION BRANCH (Score: 0.4594)
Fluency               : 0.4631
Relevance             : 1.0000
Consensus             : 0.8503
Structural Integrity  : 0.8889
Entity Density        : 0.4545
Contradiction         : 0.0000
Numeric Penalty       : 0.6667
Semantic Drift        : 0.1862
Uncertainty Multiplier: 0.6891x
Safety Compression    : 1.00x

--- CHAMPION TEXT ---
AI stands for "Artificial Intelligence." It refers to the simulation of human intelligence in machines that are programmed to think and act like humans. AI inv

# System Summary & Architectural Insights
## Stage 2 Actual Log Evaluation

Stage 2 telemetry reveals the behavioral characteristics of a compact instruction-tuned language model operating under inference-time verification controls.

The evaluation framework successfully measures multiple dimensions of generation quality, including fluency, topical alignment, semantic agreement across reasoning branches, and internal contradiction detection. However, the observed outputs demonstrate that these mechanisms **primarily assess internal coherence rather than factual correctness**.

The system consistently identified when candidate trajectories remained stable and mutually reinforcing. Yet several champion branches contained significant factual inaccuracies despite exhibiting excellent structural quality, strong semantic consensus, and negligible contradiction penalties.

> ⚠️ **Critical Operational Principle**
> Reliability signals derived from the model's own generation dynamics should not be interpreted as evidence of truth.

---

### 📈 1. Stability Metrics Capture Confidence, Not Accuracy

Across multiple prompts, `Semantic Drift` remained within `STABLE_CONVERGENCE` and `MILD_DIVERGENCE` bands, indicating that the model generated highly consistent reasoning trajectories.

However, these stable regions frequently contained incorrect factual representations, including:
* **Historical fabrications**
* **Scientific misconceptions**
* **Mathematical inaccuracies**
* **Biographical distortions**

The results demonstrate that compact language models may confidently converge on incorrect internal representations without exhibiting significant epistemic instability.

> **Key Observation:** Low semantic drift reflects generation consistency rather than factual validity.

---

### 🤝 2. Consensus Mechanisms Remain Vulnerable to Shared Hallucinations

The strongest limitation exposed by Stage 2 was the emergence of **consensus hallucinations**, where independently sampled branches converged toward identical incorrect conclusions.

High Consensus scores combined with minimal Contradiction penalties indicate that the system effectively measures agreement between reasoning trajectories but cannot determine whether those trajectories are objectively correct.

In these scenarios:
* Multiple branches reinforced the same misconceptions.
* NLI-based contradiction detection remained inactive.
* Champion selection favored coherent but incorrect outputs.

> **Key Observation:** Agreement between model-generated candidates should be treated as a measure of internal consistency rather than evidence of truth.

---

### 🎯 3. Fluency and Relevance Signals Reward Persuasive Presentation

The Fluency and Relevance axes consistently promoted responses exhibiting:
* Clear instructional formatting
* Strong alignment with prompt intent
* High linguistic confidence
* Well-organized explanatory structures

Despite these strengths, several highly ranked responses contained substantial factual errors. The evaluation demonstrates that a **persuasive communication style can mask underlying deficiencies** in knowledge representation.

> **Key Observation:** Outputs can simultaneously be highly fluent, highly relevant, and highly coherent while remaining factually incorrect.

---

### 📌 Operational Conclusion

Stage 2 functions effectively as an internal coherence and stability assessment layer. It provides valuable visibility into:
* Generation consistency
* Semantic disagreement across candidate trajectories
* Structural degradation
* Contradictory reasoning patterns
* Escalation triggers for uncertain outputs

However, the observed telemetry confirms that **inference-time agreement alone is insufficient for establishing factual reliability**.

> 💡 **Enterprise Deployment Takeaway**
> Operational trust in LLM systems must emerge from layered verification architectures rather than reliance on model confidence, fluency, or consensus signals alone.

Stage 2 successfully identifies unstable reasoning trajectories and supports uncertainty-aware routing decisions. At the same time, it exposes the necessity of independent factual validation mechanisms capable of evaluating outputs against external deterministic references.

**Looking Ahead:** This naturally motivates the introduction of **Stage 3: Factual Audit and Ground-Truth Verification**, designed to distinguish internally coherent generations from objectively correct ones.

# Stage 3 Architecture: Asynchronous Factual Audit System & NLI Verification

While Stage 2 acts as a world-class editor—grading structural, semantic, and consensus alignments within the cohort—**Stage 3: GROUND_TRUTH_REGISTRY** serves as the ultimate epistemic anchor. Executed as an asynchronous post-processing validation layer, its single mission is to subject the top-scoring surviving branches to a ruthless factual audit, piercing the veil of high-confidence, fluent hallucinations.

---

## 🎯 1. The Ground Truth Registry & Adversarial Defenses

The system bypasses typical keyword matching in favor of deep semantic parsing by mapping text fragments against a hardcoded verification layer. The registry handles entities through two distinct defensive postures:

### 🔹 Exclusive Precedence Traps (`is_trap: True`)
Designed specifically to neutralize adversarial injections and active user traps (e.g., *"Why did Albert Einstein invent the internet in 1969?"*). These profiles are evaluated with top-priority precedence. If an NLI alignment matches a forbidden claim in a trap, the system bypasses standard profiling and applies catastrophic scoring penalties immediately.

### 🔹 Standard Entity & Concept Profiles (`is_trap: False`)
Traditional factual baselines categorized by type (`entity_factual`, `conceptual`). Rather than analyzing the essay as a monolithic block, Stage 3 leverages a robust regex engine to split candidate text into distinct sentence vectors, evaluating each atomic assertion independently against two strict claim arrays:
* **Required Claims:** Vital pillars or conceptual anchors that the candidate text *must* entail to satisfy completeness.
* **Forbidden Claims:** Core historical falsehoods, logical inversions, or common small-model hallucination patterns that instantly invalidate the text's integrity.

---

## 🧮 2. The Mathematical Scoring Engine

The execution layer uses a zero-tolerance mathematical matrix to strip away credit from beautifully formatted lies. The system calculates three core metrics using Natural Language Inference (NLI) sentence-pair checks:

### 📊 Completeness Score
Measures the coverage of necessary factual pillars. It computes the ratio of matched required claims to total required claims:

$$\text{Completeness} = \frac{|\text{Matched Required Claims}|}{\max(|\text{Required Claims}|, 1)}$$

### 🚨 Integrity Score (Hard Binary Penalty)
Acts as a zero-tolerance tripwire. If an atomic sentence within the response is found to entail any forbidden claim with an NLI confidence threshold of $\ge 0.75$, the integrity multiplier collapses to absolute zero:

$$\text{Integrity} = \begin{cases} 1.0 & \text{if } |\text{Triggered Forbidden Claims}| = 0 \\ 0.0 & \text{if } |\text{Triggered Forbidden Claims}| \ge 1 \end{cases}$$

### 🏆 Combined Factual Truth Score
The final arbiter of truth. Because the metrics are coupled multiplicatively, an uncompromised text scales purely on its informational completeness, whereas a single elegant lie completely zeros out the entire evaluation track:

$$\text{Truth Score} = \text{Completeness} \times \text{Integrity}$$

---

## 📡 3. Runtime Verification & The Epistemic Gap

The execution layer loops over the pre-sorted candidate branches exported from Stage 2, creating a direct, dual-channel comparison between **Internal Confidence** and **Objective Reality**:

In [5]:
import re
import numpy as np

# ---------------------------------------------------------------------------
# UPDATED GROUND TRUTH REGISTRY (Unified Schemas)
# ---------------------------------------------------------------------------
GROUND_TRUTH_REGISTRY = {
    "einstein_internet": {
        "is_trap": True,
        "aliases": ["einstein invent the internet", "einstein internet", "einstein net", "internet in 1969"],
        "type": "correction",
        "required_claims": ["Einstein did not invent the internet", "The internet was not created by Einstein"],
        "forbidden_claims": ["einstein invented the internet", "einstein created the internet", "einstein net"]
    },
    "hilbert": {
        "is_trap": False,
        "aliases": ["david hilbert", "hilbert's"],
        "type": "entity_factual",
        "required_claims": ["David Hilbert was a German mathematician", "David Hilbert died in 1943"],
        "forbidden_claims": ["won the nobel prize", "invented set theory", "born in goettingen", "died in berlin"]
    },
    "einstein": {
        "is_trap": False,
        "aliases": ["albert einstein", "einstein's"],
        "type": "entity_factual",
        "required_claims": ["Albert Einstein was a physicist", "developed relativity"],
        "forbidden_claims": ["father was alfred", "born on april 14", "painter and composer", "wrote music for operas", "university of munich"]
    },
    "baudrillard": {
        "is_trap": False,
        "aliases": ["jean baudrillard", "baudrillard's"],
        "type": "entity_factual",
        "required_claims": ["Jean Baudrillard was a French sociologist", "wrote about simulation and hyperreality"],
        "forbidden_claims": ["grew up in italy", "university of rome", "消费ism", "book in 1980"]
    },
    "ai": {
        "is_trap": False,
        "aliases": ["artificial intelligence", "what is ai"],
        "type": "conceptual",
        "required_claims": ["simulation of human intelligence", "learn from data"], # Fixed: unified 'anchors' to 'required_claims'
        "forbidden_claims": ["conscious like humans", "single algorithm"]
    }
}

# ---------------------------------------------------------------------------
# SENTENCE SPLITTER
# ---------------------------------------------------------------------------
def split_into_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 5]

# ---------------------------------------------------------------------------
# ROBUST NLI ENTAILMENT CHECKER (Returns boolean and raw score)
# ---------------------------------------------------------------------------
def check_nli(nli_pipeline, premise: str, hypothesis: str, threshold=0.75):
    try:
        res = nli_pipeline(premise, text_pair=hypothesis)[0]
        label = res.get("label", "").lower()
        score = res.get("score", 0.0)
        is_entail = (label == "entailment") and (score >= threshold)
        return is_entail, score
    except Exception:
        return False, 0.0

# ---------------------------------------------------------------------------
# INSTRUMENTED STAGE 3 EVALUATOR
# ---------------------------------------------------------------------------
def run_stage3_evaluator(candidate_text, prompt_string, nli_pipeline, branch_debug_label=""):
    prompt_lower = prompt_string.lower()
    matched_key = None
    
    for key, gt in GROUND_TRUTH_REGISTRY.items():
        aliases = gt.get("aliases", [])
        if any(alias in prompt_lower for alias in aliases):
            matched_key = key
            break

    if matched_key is None:
        return {
            "type": "unverified", "completeness": 0.0, "integrity": 1.0, "truth_score": 0.5,
            "matched_positive": [], "missing_positive": [], "triggered_forbidden": [],
            "note": "No ground truth key matched; Stage 3 skipped."
        }

    gt = GROUND_TRUTH_REGISTRY[matched_key]
    sentences = split_into_sentences(candidate_text)
    required_claims = gt.get("required_claims", [])
    forbidden_claims = gt.get("forbidden_claims", [])

    matched_positive = []
    missing_positive = []
    
    # 1. REQUIRED CLAIM COVERAGE WITH VERBOSE LOGGING
    for claim in required_claims:
        found = False
        for sent in sentences:
            is_entail, nli_score = check_nli(nli_pipeline, sent, claim)
            if is_entail:
                print(f"   [DEBUG ENTAILMENT] Branch: {branch_debug_label}")
                print(f"     └─ Sentence: \"{sent}\"")
                print(f"     └─ Verified Target: \"{claim}\" (Score: {nli_score:.4f})")
                matched_positive.append(claim)
                found = True
                break
        if not found:
            missing_positive.append(claim)

    completeness = len(matched_positive) / max(len(required_claims), 1)

    # 2. FORBIDDEN CLAIM DETECTION WITH REAL DYNAMIC SCORES
    triggered_forbidden = []
    for bad_claim in forbidden_claims:
        for sent in sentences:
            is_entail, nli_score = check_nli(nli_pipeline, sent, bad_claim)
            if is_entail:
                print(f"   [DEBUG VIOLATION] Branch: {branch_debug_label}")
                print(f"     └─ Sentence: \"{sent}\"")
                print(f"     └─ Triggered Forbidden: \"{bad_claim}\" (Score: {nli_score:.4f})")
                triggered_forbidden.append((sent, bad_claim, nli_score))

    integrity = 1.0 if len(triggered_forbidden) == 0 else 0.0
    truth_score = completeness * integrity

    return {
        "type": gt.get("type", "unknown"),
        "completeness": completeness,
        "integrity": integrity,
        "truth_score": truth_score,
        "matched_positive": matched_positive,
        "missing_positive": missing_positive,
        "triggered_forbidden": triggered_forbidden
    }

# ===========================================================================
# RUNTIME AUDIT PIPELINE EXECUTION WITH DETERMINISTIC ORCHESTRATION ROUTING
# ===========================================================================
print("="*75)
print("🎯 DECOUPLED RUNTIME: STAGE 3 POST-GENERATION FACTUAL AUDIT SYSTEM")
print("="*75)

if 'pipeline_results' not in locals() and 'pipeline_results' not in globals():
    print("❌ Error: pipeline_results data dictionary structure cannot be found.")
else:
    for raw_prompt, data in pipeline_results.items():
        candidates = data.get("candidates", [])
        intent_profile = data.get("intent_profile", "⚖️ General Balanced Mode")
        semantic_drift = data.get("semantic_drift", 0.0)
        
        if not candidates:
            continue
            
        uncertainty_multiplier = np.exp(-2.0 * semantic_drift)
        print("\n" + "="*75)
        print(f"📊 HISTORICAL STAGE 2 ENGINE TELEMETRY MATRIX FOR: '{raw_prompt}'")
        print(f"-> Active Intent Profile     : {intent_profile}")
        print(f"-> Semantic Divergence Drift : {semantic_drift:.4f} (Multiplier: {uncertainty_multiplier:.4f}x)")
        print("-" * 75)
        
        best_candidate = candidates[0]
        
        # Define candidate mapping
        loop_candidates = {"🥇 CHAMPION BRANCH": best_candidate}
        if len(candidates) > 1:
            loop_candidates["🥈 CONTENDER BRANCH"] = candidates[1]

        # 1. EVALUATION PHASE: Process all branches and cache reports
        branch_reports = {}
        for branch_name, candidate in loop_candidates.items():            
            raw_text = candidate.get("text", "")
            
            # Run evaluator with real-time debug tracking onto console
            print(f"\n🔍 Auditing {branch_name}...")
            report = run_stage3_evaluator(raw_text, raw_prompt, nli_pipeline, branch_debug_label=branch_name)
            
            # Store report data for our routing gate
            # FIXED: Fallback chain ensures we capture the correct score key
            branch_score = candidate.get('score', candidate.get('final_calibrated_score', 0.0))
            
            branch_reports[branch_name] = {
                "report": report,
                "text": raw_text,
                "score": branch_score
            }
            
        # 2. ORCHESTRATION PHASE: Apply deterministic routing rules
        print("\n" + "═"*75)
        print("🤖 AUTOMATED ORCHESTRATION LAYER DECISION MATRIX")
        print("═"*75)
        
        champ_data = branch_reports["🥇 CHAMPION BRANCH"]
        champ_rep = champ_data["report"]
        
        final_output_stream = ""
        routing_reason = ""
        
        # Check if we have a two-branch system to compare
        if "🥈 CONTENDER BRANCH" in branch_reports:
            cont_data = branch_reports["🥈 CONTENDER BRANCH"]
            cont_rep = cont_data["report"]
            
            # RULE 1: Both pass integrity checks -> Rank based on highest Completeness
            if champ_rep["integrity"] == 1.0 and cont_rep["integrity"] == 1.0:
                if cont_rep["completeness"] > champ_rep["completeness"]:
                    final_output_stream = cont_data["text"]
                    routing_reason = (f"🔄 OVERRIDE ROUTING: Both branches preserved integrity, "
                                      f"but CONTENDER won out on completeness "
                                      f"({cont_rep['completeness']*100:.1f}% vs {champ_rep['completeness']*100:.1f}%).")
                else:
                    final_output_stream = champ_data["text"]
                    routing_reason = "✅ MAINTAIN CHAMPION: Both branches preserved integrity. Champion maintained factual superiority/parity."
            
            # RULE 2: Safe Routing -> One passes, one fails
            elif champ_rep["integrity"] != cont_rep["integrity"]:
                if champ_rep["integrity"] == 1.0:
                    final_output_stream = champ_data["text"]
                    routing_reason = "✅ MAINTAIN CHAMPION: Champion preserved absolute factual integrity while Contender collapsed into hallucination."
                else:
                    final_output_stream = cont_data["text"]
                    routing_reason = "🔄 OVERRIDE ROUTING: Champion failed critical integrity benchmarks! Contender cleanly routed to protect production stream."
            
            # RULE 3: Catastrophic Double-Fail -> Strip contents and return hard fallback message
            elif champ_rep["integrity"] == 0.0 and cont_rep["integrity"] == 0.0:
                final_output_stream = "⚠️ I am unable to verify the historical or factual accuracy of the claims required to fulfill this response."
                routing_reason = "❌ CATASTROPHIC SYSTEM FAILURE: Both generation channels triggered zero-integrity metrics due to severe hallucinations. Hard fallback deployed."
        
        else:
            # Single branch fallback rule block
            if champ_rep["integrity"] == 0.0:
                final_output_stream = "⚠️ I am unable to verify the historical or factual accuracy of the claims required to fulfill this response."
                routing_reason = "❌ SINGLE BRANCH BREAK: Lone champion branch failed critical integrity benchmarks. Hard fallback deployed."
            else:
                final_output_stream = champ_data["text"]
                routing_reason = "✅ PASS-THROUGH: Single branch verification confirmed stable integrity parameters."

        # 3. PRODUCTION STREAM DELIVERY
        print(f"-> Decision Verdict : {routing_reason}")
        print("-" * 75)
        print("🚀 FINAL PRODUCTION STREAM OUTPUT:")
        print("-" * 75)
        print(final_output_stream.strip())
        print("=" * 75 + "\n")

🎯 DECOUPLED RUNTIME: STAGE 3 POST-GENERATION FACTUAL AUDIT SYSTEM

📊 HISTORICAL STAGE 2 ENGINE TELEMETRY MATRIX FOR: 'What is AI?'
-> Active Intent Profile     : ⚖️ General Balanced Mode
-> Semantic Divergence Drift : 0.1862 (Multiplier: 0.6891x)
---------------------------------------------------------------------------

🔍 Auditing 🥇 CHAMPION BRANCH...
   [DEBUG ENTAILMENT] Branch: 🥇 CHAMPION BRANCH
     └─ Sentence: "AI stands for "Artificial Intelligence." It refers to the simulation of human intelligence in machines that are programmed to think and act like humans."
     └─ Verified Target: "simulation of human intelligence" (Score: 0.9849)
   [DEBUG ENTAILMENT] Branch: 🥇 CHAMPION BRANCH
     └─ Sentence: "AI involves creating algorithms and models that can learn from data, recognize patterns, and make decisions or take actions based on those patterns."
     └─ Verified Target: "learn from data" (Score: 0.9366)

🔍 Auditing 🥈 CONTENDER BRANCH...
   [DEBUG ENTAILMENT] Branch: 🥈 CONTE

# Stage 3 Runtime Verification Report
## Epistemic Gap Analysis: Internal Confidence vs Objective Reality

### Overview
Stage 3 successfully demonstrated the necessity of introducing an external factual verification layer operating independently from the Stage 2 ranking matrix. 

While Stage 2 effectively selected outputs exhibiting high fluency, prompt alignment, and internal consistency, Stage 3 exposed multiple scenarios where these same high-ranking responses contained severe factual inaccuracies.

> 💡 **Core Case Study Hypothesis**
> Statistical confidence is not equivalent to factual correctness.

Stage 3 transformed the system from selecting the most *internally convincing* response into selecting the most *externally defensible* response according to deterministic constraints.

---

## 1. Major Architectural Successes

### 1.1 Severe Hallucinations Were Successfully Intercepted
The Ground Truth Registry correctly identified high-confidence factual failures that Stage 2 alone could not detect.

#### 🔍 Case Study: Einstein Internet Trap
* **Stage 2 Assessment:**
  * Moderate semantic instability (`Drift = 0.5151`)
  * Champion branch selected despite acknowledging the false premise
  * Internal confidence mechanisms alone failed to reject the fabricated scenario
* **Stage 3 Detection:**
  * ❌ Detected forbidden claim: *"einstein invented the internet"*
  * ❌ Detected forbidden claim: *"einstein created the internet"*
* **Result:** `Integrity Score = 0.0`, `Truth Score = 0.0` → **Champion branch rejected**; safe contender branch routed to production.

> **Interpretation:** Consensus does not imply truth. Small models may partially accommodate false premises, but registry-based verification successfully defends against adversarial prompt injections.

---

### 1.2 Historical Fabrications Prevented From Reaching Production
#### 🔍 Case Study: David Hilbert
* **Stage 2 Assessment:**
  * Extremely low semantic drift (`0.0920`)
  * High internal confidence & strong branch agreement
  * Generated hallucinations included: *Hilbert winning a Nobel Prize* and incorrect biographical details.
* **Stage 3 Detection:**
  * **Champion:** ✔️ Verified: German mathematician | ✔️ Verified: Died in 1943 | ❌ Forbidden: *"won the nobel prize"*
  * **Contender:** ✔️ Verified: German mathematician | ✔️ Verified: Died in 1943 | ❌ Forbidden: *"born in goettingen"*
* **Result:** Both branches failed integrity validation → **Hard fallback response deployed**.

> **Interpretation:** Stage 3 successfully prevented highly coherent historical misinformation from entering the final output stream. Low epistemic drift cannot be interpreted as evidence of factual reliability.

---

### 1.3 Verification Improved Branch Selection Quality
Stage 3 did not merely reject unsafe outputs; it actively optimized and improved production responses.

| Target Entity | Branch | Evaluation Metrics | Action / Outcome |
| :--- | :--- | :--- | :--- |
| **Albert Einstein** | Champion<br>Contender | Completeness: 50% \| Integrity: Preserved<br>Completeness: 100% \| Integrity: Preserved | **Contender promoted** to production. |
| **Jean Baudrillard** | Champion<br>Contender | Completeness: 0%<br>Completeness: 50% (Verified core biographical facts) | **Contender promoted** to production. |
| **Artificial Intelligence** | Champion<br>Contender | Completeness: 100% (Verified core definition)<br>Equivalent factual coverage | **Champion preserved**. |

> **Interpretation:** Stage 3 functions simultaneously as a safety layer that suppresses hallucinations and a quality optimization layer that promotes stronger factual coverage.

---

## 2. Epistemic Gap Observations

The experiments revealed a persistent gap between internal model generation dynamics and factual reality:

* **Stage 2 Internal Confidence Measures:** Fluency, relevance, consensus, structural coherence, and cross-branch contradiction.
* **Stage 3 Objective Reality Measures:** Required claim coverage, forbidden claim violations, deterministic factual constraints, and external verification signals.

### Epistemic Gap Matrix

| Prompt | Stage 2 Assessment | Stage 3 Outcome |
| :--- | :--- | :--- |
| **AI** | Strong | Champion maintained |
| **Einstein (Biography)** | Strong | Contender promoted |
| **David Hilbert** | Strong | Hard fallback triggered |
| **Einstein Internet Trap** | Moderate | Champion overridden |
| **Baudrillard** | Strong | Contender promoted |
| **Geometry** | Strong | Passed (no registry coverage) |
| **Entropy** | Strong | Passed (no registry coverage) |
| **London** | Strong | Passed (no registry coverage) |
| **Love** | Strong | Passed (no registry coverage) |
| **Large Language Models** | Strong | Passed (no registry coverage) |

---

## 3. Remaining Limitations Revealed by Stage 3

### ⚠️ 3.1 Registry Coverage Bottleneck
Several prompts bypassed factual auditing because corresponding registry profiles did not exist (e.g., *Geometry, Entropy, London, Love, Large Language Models*).
* **Takeaway:** Stage 3 can only validate domains for which explicit factual anchors have been defined. The absence of violations should not be interpreted as proof of correctness.

### ⚠️ 3.2 Dependence on NLI Classification Quality
The factual audit system relies heavily on NLI entailment judgments. While effective overall, the approach remains susceptible to:
* Threshold sensitivity
* Semantic ambiguity
* Occasional over-triggering or under-triggering
* **Takeaway:** NLI serves as a strong verification component, but it is not an infallible fact-checking mechanism.

### ⚠️ 3.3 Verification Remains Domain-Bounded
The architecture demonstrates that deterministic verification systems improve reliability within explicitly modeled domains.
* **Takeaway:** Novel claims, unseen entities, and complex scientific explanations still require broader factual grounding mechanisms. Stage 3 improves trustworthiness without claiming universal factual validation.

---

## 4. What Stage 3 Actually Achieved

Stage 3 fundamentally shifted the pipeline pipeline perspective:
* **Old Pipeline Goal:** *"Which response appears strongest internally?"*
* **Stage 3 Pipeline Goal:** *"Which response satisfies external factual constraints?"*

**Key Architectural Features Introduced:**
* Claim-level entailment auditing
* Registry-based factual anchoring
* Adversarial prompt defenses
* Integrity-based routing decisions
* Production fallback mechanisms
* Champion branch overrides
* Deterministic verification controls

---

## 5. Key Finding & Final Conclusion

The complete pipeline demonstrates a critical property of operational LLM systems:

$$\text{Fluent } \neq \text{ Correct}$$
$$\text{Relevant } \neq \text{ Correct}$$
$$\text{Consistent } \neq \text{ Correct}$$
$$\text{Consensus } \neq \text{ Correct}$$

Only external verification against independent constraints can successfully identify **stable hallucinations**, **consensus-driven misinformation**, and **high-confidence factual failures**. Stage 3 functions as the system's *Epistemic Safety Layer*, bridging the gap between internal model confidence and objective reality.

> 📌 **Final Architectural Takeaway**
> A language model can be simultaneously fluent, relevant, internally consistent, and highly confident while still being factually incorrect. 
> 
> * **Stage 2** measures how strongly the model believes itself.
> * **Stage 3** evaluates whether independent constraints support those beliefs.
> 
> Together, these layers form a practical reliability framework illustrating how deterministic controls, external validation mechanisms, and controlled escalation pathways can improve trustworthiness in probabilistic AI systems without modifying the underlying model itself.